In [ ]:
import pandas as pd
import numpy as np
import pyreadr
import directories as dir
import matplotlib.pyplot as plt

# Create directory for data
data_fs = dir.ROOT / "data" / "raw_data" / "fs.csv"
data_suvr = dir.ROOT / "data" / "raw_data" / "suvr.csv"

# Parse csv
fs = pd.read_csv(data_fs)
suvr = pd.read_csv(data_suvr)

"""
----- Freesurfer (FS) data exploration -----
Key:
- CV = cortical volume 
- SA = surface area 
- TA = cortical thickness avg 
- TS = thickness stdev 
- SV = subcortical volume
"""

print("----- FS data exploration -----")
print("Shape: ", fs.shape)
print("Unique subjects (RID): ", fs['RID'].nunique())
print("QC total counts: \n", fs['OVERALLQC'].value_counts())

st_cols = [c for c in fs.columns if c.startswith("ST")] 
cv_cols = [c for c in st_cols if c.endswith("CV")] 
sa_cols = [c for c in st_cols if c.endswith("SA")] 
ta_cols = [c for c in st_cols if c.endswith("TA")] 

print("ST count: ", len(st_cols))
print("CV count: ", len(cv_cols))
print("SA count: ", len(sa_cols))
print("TA count: ", len(ta_cols))

----- Freesurfer data exploration -----
Shape:  (12167, 347)
Unique subjects (RID):  3175
QC total counts: 
 OVERALLQC
Partial             586
Pass                478
Hippocampus Only     11
Fail                  6
Name: count, dtype: int64
ST count:  326
CV count:  69
SA count:  69
TA count:  68


/var/folders/rt/m33zscwn3p9700c8mctz1h0w0000gn/T/ipykernel_29319/2397179679.py:12: DtypeWarning: Columns (0: OVERALLQC, 1: TEMPQC, 2: FRONTQC, 3: PARQC, 4: INSULAQC, 5: OCCQC, 6: BGQC, 7: CWMQC, 8: VENTQC, 9: HIPPOQC) have mixed types. Specify dtype option on import or set low_memory=False.
  fs = pd.read_csv(data_fs)


In [72]:
"""
----- PET Standardized Uptake Value Ratio (SUVR) data exploration -----
QC Results Key:
- 2  = pass
- 1  = partial pass
- 0  = fail
- -1 = not assessed
- -2 = cannot be processed 
"""

print("\n----- PET SUVR data exploration -----")
print("Shape: ", suvr.shape)
print("Unique subjects (RID): ", suvr['RID'].nunique())
print("QC total counts: \n", suvr['qc_flag'].value_counts())
print("QC = 2 counts: ", suvr[suvr['qc_flag'] == 2]['RID'].nunique())
print("QC >= 1 counts: ", suvr[suvr['qc_flag'] >= 1]['RID'].nunique())
print("QC >= 0 counts: ", suvr[suvr['qc_flag'] >= 0]['RID'].nunique())


----- PET SUVR data exploration -----
Shape:  (2489, 339)
Unique subjects (RID):  1448
QC total counts: 
 qc_flag
 2    2311
-1     137
 1      30
-2       9
 0       2
Name: count, dtype: int64
QC = 2 counts:  1375
QC >= 1 counts:  1386
QC >= 0 counts:  1388


In [87]:
"""
----- ADNI Merge data exploration ----- 
"""

print("\n----- ADNI Merge data exploration -----")
result = pyreadr.read_r(dir.ROOT / "data" / "raw_data" / "ADNIMERGE2" / "data" / "DXSUM.rda")
dxsum = result["DXSUM"]

print(dxsum.head())



----- ADNI Merge data exploration -----
  ORIGPROT COLPROT        PTID  RID VISCODE VISCODE2    EXAMDATE DIAGNOSIS  \
0    ADNI1   ADNI1  011_S_0002  2.0      bl       bl  2005-09-29        CN   
1    ADNI1   ADNI1  011_S_0003  3.0      bl       bl  2005-09-30  Dementia   
2    ADNI1   ADNI1  011_S_0005  5.0      bl       bl  2005-09-30        CN   
3    ADNI1   ADNI1  011_S_0008  8.0      bl       bl  2005-09-30        CN   
4    ADNI1   ADNI1  022_S_0007  7.0      bl       bl  2005-10-06  Dementia   

  DXNORM DXNODEP  ... DXODES              DXCONFID    ID SITEID    USERDATE  \
0    Yes     NaN  ...    NaN      Highly Confident   2.0  107.0  2005-10-01   
1    NaN     NaN  ...    NaN  Moderately Confident   4.0  107.0  2005-10-01   
2    Yes     NaN  ...    NaN      Highly Confident   6.0  107.0  2005-10-01   
3    Yes     NaN  ...    NaN  Moderately Confident   8.0  107.0  2005-10-01   
4    NaN     NaN  ...    NaN      Highly Confident  10.0   10.0  2005-10-06   

  USERDATE2 DD_

In [ ]:
"""
----- Total Usable FS Data ----- 
Freesurfer Filter:
- OVERALL QC: Only exclude "Fail"
- FIELD_STRENGTH: 3T 
"""

print("\n----- Total Usable FS Data -----")
print("FS shape before filter: ", fs.shape[0])

# QC Filter
fs_qc = fs[
    fs["OVERALLQC"].isin(["Pass", "Partial", "Hippocampus Only"]) | 
    fs["OVERALLQC"].isna()
    ]
print("FS shape after filtering for QC: ", fs_qc.shape[0])

# FS Filter
fs_filtered = fs_qc[fs_qc["FIELD_STRENGTH"] == "3T"]
print("FS shape after filtering for FS: ", fs_filtered.shape[0])

# NaN Filter (just for fun, but will implement once FS and SUVR is merged)
st_cols = [c for c in fs_qc_fs.columns if c.startswith("ST")]
cv_cols = [c for c in st_cols if c.endswith("CV")]
sa_cols = [c for c in st_cols if c.endswith("SA")]
ta_cols = [c for c in st_cols if c.endswith('TA')]
cols = cv_cols + sa_cols + ta_cols

"""
----- Total Usable SUVR Data ----- 
SUVR Filter:
- qc_flag: Keep Pass (2) and Partial Pass (1)
- Tracer: Only flortaucipir (FTP)
"""

print("\n----- Total Usable SUVR Data -----")
print("SUVR shape before filter: ", suvr.shape[0])

# qc_flag filter
suvr_qc = suvr[suvr["qc_flag"] >= 1]
print("SUVR shape after filtering for qc_flag: ", suvr_qc.shape[0])

# Tracer filter
suvr_filtered = suvr_qc[suvr_qc["TRACER"] == "FTP"]
print("SUVR shape after filtering for tracer: ", suvr_filtered.shape[0])

"""
----- DXSUM Exploration ----- 
"""
print("\n----- DXSUM -----")
dxsum["RID"] = dxsum["RID"].astype(float).astype(int)
print("Shape: ", dxsum.shape)
print("Unique subjects: ", dxsum['RID'].nunique())
print("Diagnosis counts:\n", dxsum['DIAGNOSIS'].value_counts(dropna=False))


----- Total Usable FS Data -----
FS shape before filter:  12167
FS shape after filtering for QC:  12161
FS shape after filtering for FS:  7681

----- Total Usable SUVR Data -----
SUVR shape before filter:  2489
SUVR shape after filtering for qc_flag:  2341
SUVR shape after filtering for tracer:  2146

----- DXSUM -----
Shape:  (15881, 42)
Unique subjects:  3788
Diagnosis counts:
 DIAGNOSIS
MCI         6565
CN          6275
Dementia    2996
NaN           45
Name: count, dtype: int64


In [94]:
# ---- Merge ----
print("\n----- Merge -----")

# FS + SUVR inner join
paired = pd.merge(fs_filtered, suvr_filtered, on=["RID", "VISCODE"], how="inner")
print(f"After FS+SUVR merge: {paired['RID'].nunique()} subjects, {len(paired)} rows")

# Add diagnosis
paired = pd.merge(
    paired,
    dxsum[["RID", "VISCODE", "DIAGNOSIS"]],
    on=["RID", "VISCODE"],
    how="left"
)
print(f"Missing diagnosis: {paired['DIAGNOSIS'].isna().sum()}")
print(f"Diagnosis distribution:\n{paired['DIAGNOSIS'].value_counts()}")

# Sanity check - scan date proximity
paired["EXAMDATE"] = pd.to_datetime(paired["EXAMDATE"])
paired["SCANDATE"] = pd.to_datetime(paired["SCANDATE"])
diff_days = (paired["SCANDATE"] - paired["EXAMDATE"]).dt.days.abs()
print(f"\nMean days between MRI and PET: {diff_days.mean():.1f}")
print(f"Max days between MRI and PET: {diff_days.max():.1f}")

# Deduplicate - keep earliest PET scan per subject
paired = paired.sort_values("SCANDATE").groupby("RID").first().reset_index()
print(f"\nAfter deduplication: {paired['RID'].nunique()} subjects")
print(f"Final diagnosis distribution:\n{paired['DIAGNOSIS'].value_counts()}")

# ---- NaN Check on Paired Table ----
print("\n----- NaN Check -----")

# FS feature NaN
feat_cols = cv_cols + sa_cols + ta_cols
fs_nan_frac = paired[feat_cols].isna().mean(axis=1)
print(f"Subjects with >20% FS feature NaN: {(fs_nan_frac > 0.20).sum()}")

# SUVR target NaN
suvr_target_cols = [c for c in paired.columns if c.endswith("_SUVR")
                    and "META" not in c
                    and "INFERIORCEREBELLUM" not in c]
suvr_nan_frac = paired[suvr_target_cols].isna().mean(axis=1)
print(f"Subjects with >20% SUVR target NaN: {(suvr_nan_frac > 0.20).sum()}")

# Apply NaN filters
# Apply FS NaN filter first
paired = paired[fs_nan_frac <= 0.20].copy()

# Recompute SUVR NaN frac on the already-filtered table
suvr_nan_frac = paired[suvr_target_cols].isna().mean(axis=1)
paired = paired[suvr_nan_frac <= 0.20].copy()

print(f"Final subjects: {paired['RID'].nunique()}")
print(f"Final diagnosis:\n{paired['DIAGNOSIS'].value_counts()}")

# ---- SUVR Distribution Sanity Check ----
print("\n----- SUVR Sanity Check -----")
print(f"SUVR min: {paired[suvr_target_cols].min().min():.3f}")
print(f"SUVR max: {paired[suvr_target_cols].max().max():.3f}")
print(f"SUVR mean: {paired[suvr_target_cols].mean().mean():.3f}")
print(f"SUVR NaN remaining: {paired[suvr_target_cols].isna().sum().sum()}")


----- Merge -----
After FS+SUVR merge: 605 subjects, 1029 rows
Missing diagnosis: 6
Diagnosis distribution:
DIAGNOSIS
CN          603
MCI         282
Dementia    138
Name: count, dtype: int64

Mean days between MRI and PET: 27.7
Max days between MRI and PET: 635.0

After deduplication: 605 subjects
Final diagnosis distribution:
DIAGNOSIS
CN          343
MCI         179
Dementia     80
Name: count, dtype: int64

----- NaN Check -----
Subjects with >20% FS feature NaN: 36
Subjects with >20% SUVR target NaN: 0
Final subjects: 569
Final diagnosis:
DIAGNOSIS
CN          322
MCI         168
Dementia     76
Name: count, dtype: int64

----- SUVR Sanity Check -----
SUVR min: 0.227
SUVR max: 3.833
SUVR mean: 1.153
SUVR NaN remaining: 1086


/var/folders/rt/m33zscwn3p9700c8mctz1h0w0000gn/T/ipykernel_29319/1002851414.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  paired = paired.sort_values("SCANDATE").groupby("RID").first().reset_index()


In [95]:
outlier = paired[(paired["SCANDATE"] - paired["EXAMDATE"]).dt.days.abs() > 180]
print(outlier[["RID", "VISCODE", "EXAMDATE", "SCANDATE"]])

      RID VISCODE   EXAMDATE   SCANDATE
22    896    init 2018-03-15 2018-11-14
65   4067    init 2017-08-16 2018-02-20
124  4404    init 2017-12-27 2018-07-25
148  4536    init 2017-11-02 2019-01-08
161  4643    init 2017-10-03 2018-08-21
162  4644    init 2017-06-21 2018-04-17
271  6062      y2 2020-01-27 2021-01-14
272  6063      y2 2020-02-25 2021-06-01
275  6069      y2 2019-12-10 2020-07-15
287  6111  4_init 2024-10-28 2025-04-28
318  6206      y1 2019-04-30 2019-11-12
358  6314  4_init 2025-04-04 2024-09-12
372  6347      y2 2020-09-03 2021-05-26
421  6487  4_init 2025-01-20 2025-09-22
554  6951  4_init 2025-05-02 2024-10-24
570  7011  4_init 2025-07-03 2024-10-21
